# Import / Export

Central notebook for converting data between the formats used across the annotation
pipeline: **Label Studio** (annotation), **spaCy** (rule-based REX), and **GLiNER2**
(NER + zero-shot/LoRA REX).

All conversion logic lives in [`conversions.py`](./conversions.py), which sits next to
this notebook. Each code cell below just imports that module and calls the relevant
function(s) with your paths — edit the paths/filters and run.


## Index

**[Format overview](#format-overview)** — what each format looks like

**Conversions**
1. [Source text → Label Studio importable JSON](#1-source-text-to-label-studio-importable-json)
2. [Label Studio minified export → Label Studio importable JSON (with annotations)](#2-label-studio-minified-export-to-label-studio-importable-json)
3. [Label Studio full export → spaCy JSON](#3-label-studio-full-export-to-spacy-json)
4. [Label Studio full export → GLiNER JSON](#4-label-studio-full-export-to-gliner-json)
5. [GLiNER JSON → spaCy JSON](#5-gliner-json-to-spacy-json)
6. [GLiNER JSON → Label Studio importable JSON (annotations)](#6-gliner-json-to-label-studio-importable-json-annotations)
7. [GLiNER JSON → Label Studio importable JSON (predictions)](#7-gliner-json-to-label-studio-importable-json-predictions)

**Utilities**
- [U1. Wrap a dict as a list / combine a folder of JSON files](#u1-wrap-a-dict-as-a-list-or-combine-a-folder-of-json-files)
- [U2. Split a task list into one file per task](#u2-split-a-task-list-into-one-file-per-task)
- [U3. Merge Label Studio predictions + annotations files](#u3-merge-label-studio-predictions-and-annotations-files)
- [U4. Strip relation annotations from a Label Studio export](#u4-strip-relation-annotations-from-a-label-studio-export)


## Setup

In [2]:
# All conversion functions used below live in conversions.py, in the same folder
# as this notebook.
from conversions import *


## Format overview

Quick reference for what each format looks like. All examples below describe the
same sentence: *"Beatus Martinus Turonis obiit."* ("person" entity: Martinus,
"place" entity: Turonis, relation: `resides_at`).


### Label Studio minified export
The export format available on Label Studio's **free tier** (whole project only,
one flat object per task, no nested `annotations` structure).
```json
[
  {
    "id": 12008,
    "text": "Beatus Martinus Turonis obiit.",
    "annotator": 8,
    "annotation_id": 20441,
    "created_at": "2024-03-01T10:00:00Z",
    "updated_at": "2024-03-01T10:05:00Z",
    "lead_time": 42.3,
    "label": [
      {"start": 7, "end": 15, "text": "Martinus", "labels": ["person"]}
    ]
  }
]
```


### Label Studio full export (entities + relations)
The full JSON export (not minified) — the only export that includes **relations**
as well as entities. Required input for the spaCy/GLiNER conversions below.
```json
[
  {
    "id": 21194,
    "data": {"text": "Beatus Martinus Turonis obiit."},
    "annotations": [
      {
        "completed_by": 8,
        "result": [
          {"id": "e1", "type": "labels",
           "value": {"start": 7, "end": 15, "text": "Martinus", "labels": ["person"]}},
          {"id": "e2", "type": "labels",
           "value": {"start": 16, "end": 23, "text": "Turonis", "labels": ["place"]}},
          {"id": "r1", "type": "relation", "from_id": "e1", "to_id": "e2",
           "labels": ["resides_at"], "direction": "right"}
        ]
      }
    ]
  }
]
```


### spaCy JSON
Flat per-record format for rule-based spaCy REX: one object per task/text.
```json
{
  "text": "Beatus Martinus Turonis obiit.",
  "entities": [
    {"start": 7, "end": 15, "label": "person", "text": "Martinus", "id": "e1"},
    {"start": 16, "end": 23, "label": "place", "text": "Turonis", "id": "e2"}
  ],
  "relations": [
    {"head": "e1", "child": "e2", "label": "resides_at", "direction": "right"}
  ],
  "task_id": 21194
}
```


### GLiNER JSON
Flat per-record format for GLiNER2 (NER + zero-shot/LoRA REX). Very similar to the
spaCy format, but relations reference `from_id`/`to_id` instead of `head`/`child`.
```json
{
  "text": "Beatus Martinus Turonis obiit.",
  "entities": [
    {"id": "e1", "start": 7, "end": 15, "label": "person", "text": "Martinus"},
    {"id": "e2", "start": 16, "end": 23, "label": "place", "text": "Turonis"}
  ],
  "relations": [
    {"from_id": "e1", "to_id": "e2", "label": "resides_at", "direction": "right"}
  ]
}
```


### Label Studio importable JSON (predictions)
Model output (e.g. GLiNER predictions, or an LLM pre-annotation) reformatted for
Label Studio's **pre-annotation** workflow, so annotators only have to correct it
rather than label from scratch. Distinct from an *annotation* import: this is
attached under `predictions`, not `annotations`, and is tagged with a `model_version`.
```json
{
  "id": 1,
  "data": {"text": "Beatus Martinus Turonis obiit."},
  "predictions": [
    {
      "model_version": "gliner-lora-v1",
      "result": [
        {"id": "e1", "from_name": "label", "to_name": "text", "type": "labels",
         "origin": "prediction",
         "value": {"start": 7, "end": 15, "text": "Martinus", "labels": ["person"]}},
        {"id": "r1", "type": "relation", "origin": "prediction",
         "from_id": "e1", "to_id": "e2", "direction": "right", "labels": ["resides_at"]}
      ]
    }
  ]
}
```


## Conversions

### 1. Source text → Label Studio importable JSON

Splits a source text file into segments (default pattern: before each `[<number>]`
marker — adjust for other text editions) and either:
- bundles all segments into a single Label Studio importable JSON (`text_file_to_ls_json`), or
- writes each segment out as its own `.txt` file (`text_file_to_individual_files`).


In [ ]:
# ---- single file -> one LS importable JSON with one task per segment ----
text_file_to_ls_json(
    input_file="./test_texts/BHL7120-7125 - clean.txt",
    # output_file=None,               # defaults to "<stem> - LS.json" next to the input
    # pattern=r"(?=\[\d+\])",         # segment-split pattern; default splits before [<number>]
)


In [ ]:
# ---- single file -> individual .txt files, one per segment ----
text_file_to_individual_files(
    input_file="./test_texts/BHL2605 - clean.txt",
    # output_dir=None,                # defaults to "<stem>_segments/" next to the input
    # pattern=r"(?=\[\d+\])",
)


### 2. Label Studio minified export → Label Studio importable JSON

Converts a *minified* Label Studio export (the flat, free-tier export format) back
into a full importable JSON that carries the annotated entity labels. Optionally
filter by annotator and/or task id first — handy since the free tier can only
export a whole project at once.


In [ ]:
minified_ls_to_annotated_ls(
    input_path="Hagiographic-NER-training-data.json",   # a single .json file OR a folder of .json files
    # output_path=None,                # defaults to "<stem>_converted_output.json"
    from_name="label",                 # must match your labeling config: <Labels name="label" ...>
    to_name="text",                    # must match your labeling config: <Text name="text" ...>
    allowed_annotators={8, 11},        # empty set / None to skip filtering
    allowed_ids=set(),                 # e.g. {12004, 12005}; empty set / None to skip filtering
)


### 3. Label Studio full export → spaCy JSON

Converts the **full** Label Studio export (not minified — the full export is
required to get relations, not just entities) into per-task spaCy JSON records.

Optional filters: `allowed_annotators`, `allowed_task_ids`, `allowed_updated_by`
(task-level), and `allowed_entities`, `allowed_relations` (label-level — tasks left
with nothing relevant after label filtering are dropped).

Output is a folder next to the input, named `<stem>_spacy[_ents-<...>][_rels-<...>]`,
containing one JSON file per task, e.g.:
```
sample_texts/
├── relation-annotated-example.json
└── relation-annotated-example_rels-is_located_at/
    └── relation-annotated-example_task_21194.json
```


In [3]:
export_ls_tasks_to_spacy_files(
    input_path="sample_texts/relation-annotated-example.json",  # full LS export json (not minified)
    allowed_annotators={8, 11},   # None to skip; 8 = Heike, 11 = Fiametta
    allowed_task_ids=None,        # e.g. {21194, 21195}
    allowed_updated_by=None,
    allowed_entities=None,        # e.g. {"person", "place"}
    allowed_relations={"is_located_at"},       # None to skip; e.g. {"is_related_to", "goes_to"}
)


Processed 'relation-annotated-example.json': saved 4 spaCy records to 'sample_texts\relation-annotated-example_spacy_rels-is_located_at' (dropped 35 tasks with no relevant content)


WindowsPath('sample_texts/relation-annotated-example_spacy_rels-is_located_at')

### 4. Label Studio full export → GLiNER JSON

Same idea as conversion 3, but produces a single GLiNER-style JSON file
(a list of `{"text", "entities", "relations"}` records) instead of a folder of
per-task spaCy files.


In [ ]:
export_ls_tasks_to_gliner_file(
    input_path="LoRA-training/hagio_REX_training_data_53.json",
    output_path="LoRA-training/hagio_REX_training_data_53_gliner.json",
    allowed_annotators={8, 11},
    allowed_task_ids=None,
    allowed_updated_by=None,
    allowed_entities=None,        # e.g. {"person", "place"}
    allowed_relations=None,       # e.g. {"is_related_to", "resides_at"}
)


### 5. GLiNER JSON → spaCy JSON

Converts GLiNER-style items (e.g. model predictions) into the same spaCy record
shape used in conversion 3, for consistency across downstream tooling.


In [ ]:
export_gliner_to_spacy_file(
    input_path="gliner_output.json",           # your GLiNER predictions file
    output_path="gliner_output_spacy.json",
    allowed_entities=None,
    allowed_relations=None,
)


### 6. GLiNER JSON → Label Studio importable JSON (annotations)

Converts GLiNER-style items into a Label Studio import file where the entities
and relations are attached as **annotations** — i.e. treated as already-completed
work / ground truth. Use this for re-importing corrected or gold-standard data,
not for pre-annotation (see conversion 7 for that).


In [ ]:
export_gliner_to_ls_annotations_file(
    input_path="sample_texts/hagio_REX_annotation_39_spacy/combined/combined.json",
    output_path="sample_texts/hagio_REX_annotation_39_spacy/combined/annotation_combined_LS.json",
    from_name="label",   # must match your labeling config
    to_name="text",
)


### 7. GLiNER JSON → Label Studio importable JSON (predictions)

Converts GLiNER-style items (or already LS-shaped items) into Label Studio
**prediction** tasks for the pre-annotation workflow. Requires a `model_version`
label so you can tell which model/run a prediction came from once it's in
Label Studio.


In [ ]:
export_gliner_to_ls_predictions_file(
    input_path="pre-annotations/hagio_preannotation_GPT/combined/combined2.json",
    model_version="ChatGPT-pred.",   # <-- change to the name of the model you used
    # output_path=None,              # defaults to "<stem>_LS_predictions.json"
    from_name="label",
    to_name="text",
)


## Utilities

### U1. Wrap a dict as a list, or combine a folder of JSON files

Every conversion above expects a JSON **list** of tasks/items. Use this if:
- a single file holds one bare JSON object instead of a list (wraps it in `[...]`), or
- you have a folder of many single-task JSON files and want them combined into one
  list (saved to `<folder>/combined/combined.json` by default).


In [ ]:
wrap_or_combine_json(
    input_path="sample_texts/hagio_REX_annotation_39_spacy",  # a file OR a folder
    # output_path=None,   # file: overwrites in place; folder: defaults to "<folder>/combined/combined.json"
)


### U2. Split a task list into one file per task

Splits a JSON file containing a list of task objects into separate JSON files, one
per task (each task must have a `task_id` field). Files are written to a folder
named after the input file.


In [ ]:
split_tasks_to_files(
    input_path="yourfile.json",
    # output_dir=None,   # defaults to a folder named after the input file
)


### U3. Merge Label Studio predictions and annotations files

Merges a predictions JSON file and an annotations JSON file into one combined
importable JSON, matching tasks by exact text. Handy after running conversion 7
on model output and then annotating a copy: this recombines both into a single
review file.


In [ ]:
merge_predictions_and_annotations(
    predictions_path="pre-annotations/hagio_preannotation_GPT/combined/prediction_combined2_LS.json",
    annotations_path="sample_texts/hagio_REX_annotation_39_spacy/combined/annotation_combined_LS.json",
    output_path="pre-annotations/hagio_preannotation_GPT/combined/combined_predictions_annotations_LS.json",
)


### U4. Strip relation annotations from a Label Studio export

Removes all `relation` entries from a Label Studio JSON file's annotations and
predictions, keeping only entities — e.g. to prepare entity-only data for LLM
pre-annotation of relations.


In [ ]:
strip_relations(
    input_path="sample_texts/hagio_REX_annotation_53.json",
    # output_path=None,   # defaults to "<stem>_no_relations.json"
)
